# Daily Challenge - Predicting Customer Churn
## Part 1: Inferential Statistics
## Part 2: Predictive Machine Learning Model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import scipy.stats
from scipy.stats import ttest_ind
import seaborn as sns

# Machine Learning Imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

%matplotlib inline
sns.set_style("whitegrid")
matplotlib.rcParams['figure.figsize'] = (8.0, 5.0)

### 1. Data Loading and Initial Exploration

In [ ]:
## TODO : load the csv file
file_1 = 'Churn_Modelling.csv'
df = pd.read_csv(file_1)

## TODO : output the first 5 lines
display(df.head())

## TODO : Create two separate DataFrames for Churn vs Non-Churn
df_0 = df[df['Exited'] == 0] # Stayed
df_1 = df[df['Exited'] == 1] # Left

--- 
## PART 1: INFERENTIAL STATISTICS (From previous template)
### Hypothesis 1: Age

In [ ]:
## TODO: Plot the age distribution 
sns.kdeplot(df_0['Age'], fill=True, color="blue", label='Stayed')
sns.kdeplot(df_1['Age'], fill=True, color="red", label='Left')
plt.title('Age Distribution: Stayed vs Left')
plt.legend()
plt.show()

## TODO: Perform a t-test to compare the ages
t_stat, p_val = ttest_ind(df_0['Age'], df_1['Age'])
print(f"Age T-test results: T-stat = {t_stat:.4f}, P-value = {p_val:.4f}")
# Conclusion: We reject H0. Age is significantly different between groups.

### Hypothesis 2: Credit Score

In [ ]:
## TODO: Create histograms for the CreditScore distribution
plt.hist(df_0['CreditScore'], bins=30, alpha=0.5, label='Stayed', color='blue')
plt.hist(df_1['CreditScore'], bins=30, alpha=0.5, label='Left', color='red')
plt.title('Credit Score Distribution')
plt.legend()
plt.show()

## TODO: Perform a t-test
t_stat_cs, p_val_cs = ttest_ind(df_0['CreditScore'], df_1['CreditScore'])
print(f"Credit Score T-test P-value: {p_val_cs:.4f}")
# Conclusion: We reject H0, but the difference is much less pronounced than age.

### Hypothesis 3 & 4: Balance & Estimated Salary (Summarized)

In [ ]:
## T-Test for Balance
t_stat_bal, p_val_bal = ttest_ind(df_0['Balance'], df_1['Balance'])
print(f"Balance T-test P-value: {p_val_bal:.4f}")

## T-Test for Estimated Salary
t_stat_sal, p_val_sal = ttest_ind(df_0['EstimatedSalary'], df_1['EstimatedSalary'])
print(f"Estimated Salary T-test P-value: {p_val_sal:.4f}")

# Final Conclusion for Stats:
# Age and Balance are the most significant differentiators. Salary is not statistically significant.

--- 
## PART 2: MACHINE LEARNING MODEL (To address the correction report)
### Data Preprocessing & Feature Engineering

In [ ]:
# 1. Drop irrelevant columns
data_ml = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# 2. Encode categorical variables (Geography and Gender)
print("Before encoding:", data_ml[['Geography', 'Gender']].head(2).to_dict())

label_encoder = LabelEncoder()
data_ml['Gender'] = label_encoder.fit_transform(data_ml['Gender']) # 1 = Male, 0 = Female
data_ml = pd.get_dummies(data_ml, columns=['Geography'], drop_first=True) # One-hot encode Geography

print("\nAfter encoding:\n", data_ml.head(3))

### Train/Test Split & Scaling

In [ ]:
# Define Features (X) and Target (y)
X = data_ml.drop('Exited', axis=1)
y = data_ml['Exited']

# Split into 80% training and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the features (Important for models like Logistic Regression and helps Random Forest converge faster)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

### Model Training & Evaluation

In [ ]:
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = rf_model.predict(X_test_scaled)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Visualize the Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Stayed (0)', 'Exited (1)'], 
            yticklabels=['Stayed (0)', 'Exited (1)'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Random Forest')
plt.show()

In [ ]:
# Feature Importance
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
feature_names = X.columns

plt.figure(figsize=(10, 6))
plt.title("Feature Importances for Predicting Churn")
plt.bar(range(X.shape[1]), importances[indices], align="center")
plt.xticks(range(X.shape[1]), feature_names[indices], rotation=45, ha="right")
plt.tight_layout()
plt.show()